In [20]:
import os
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import librosa

In [45]:
XML_DIR = "./IDMT-SMT-DRUMS-V2/annotation_xml/"
xml_list = []

for filename in os.listdir(XML_DIR):
    if filename.endswith(".xml"):
        file_path = os.path.join(folder_path,filename)

        tree = ET.parse(file_path)
        root = tree.getroot()
    
        for event in root.findall('.//event'):
            # Extracting based on the tags seen in your code snippet
            onset = float(event.find('onsetSec').text)
            label = event.find('instrument').text
            
            xml_list.append({
                'filename': filename,
                'onset': onset,
                'label': label
            })
xml_list

[{'filename': 'RealDrum01_00#MIX.xml', 'onset': 0.08585, 'label': 'HH'},
 {'filename': 'RealDrum01_00#MIX.xml', 'onset': 0.091791, 'label': 'KD'},
 {'filename': 'RealDrum01_00#MIX.xml', 'onset': 0.50431, 'label': 'HH'},
 {'filename': 'RealDrum01_00#MIX.xml', 'onset': 0.52608, 'label': 'SD'},
 {'filename': 'RealDrum01_00#MIX.xml', 'onset': 0.94222, 'label': 'HH'},
 {'filename': 'RealDrum01_00#MIX.xml', 'onset': 0.97488, 'label': 'KD'},
 {'filename': 'RealDrum01_00#MIX.xml', 'onset': 1.37, 'label': 'HH'},
 {'filename': 'RealDrum01_00#MIX.xml', 'onset': 1.3896, 'label': 'SD'},
 {'filename': 'RealDrum01_00#MIX.xml', 'onset': 1.7814, 'label': 'HH'},
 {'filename': 'RealDrum01_00#MIX.xml', 'onset': 1.8362, 'label': 'KD'},
 {'filename': 'RealDrum01_00#MIX.xml', 'onset': 2.215, 'label': 'HH'},
 {'filename': 'RealDrum01_00#MIX.xml', 'onset': 2.2567, 'label': 'SD'},
 {'filename': 'RealDrum01_00#MIX.xml', 'onset': 2.4625, 'label': 'SD'},
 {'filename': 'RealDrum01_00#MIX.xml', 'onset': 2.6489, 'lab

In [52]:
# ------SETTINGS------
SR = 44100
HOP_SIZE = 512
N_MELS = 128
AUDIO_DIR = "./IDMT-SMT-DRUMS-V2/audio/"
SAVE_DIR = "./Processed_Data/"

if not os.path.exists(SAVE_DIR): os.makedirs(SAVE_DIR)

# 1. Get a list of all unique filenames in big_list
unique_files = set([item["filename"] for item in xml_list])

for filename in unique_files:
    print(f"Processing {filename}...")
    
    # 2. Filter the big list for just this file
    # This creates a sub-list of [time, label]
    file_annotations = [[item["onset"], item["label"]] for item in xml_list if item["filename"] == filename]
    
    # 3. Load Audio
    audio_filename = filename.replace(".xml",".wav")
    wav_path = os.path.join(AUDIO_DIR, audio_filename)
    if os.path.exists(wav_path):
        y_audio, _ = librosa.load(wav_path, sr=SR)
    else:
        print(f"Warning: could not find audio for {filename}")
        continue


    # 4. Convert to Mel Spectrogram
    spec = librosa.feature.melspectrogram(y=y_audio, sr=SR, n_fft=2048, hop_length=HOP_SIZE, n_mels=N_MELS)
    X = librosa.power_to_db(spec, ref=np.max).T # Transpose to (Frames, Bins)
    
    # 4. Create the Y (Target) Matrix [Kick, Snare, HiHat]
    Y = np.zeros((X.shape[0], 3))
    label_map = {'KD': 0, 'SD': 1, 'HH': 2}
    
    for onset_time, label in file_annotations:
        frame_idx = int(round(float(onset_time) * SR / HOP_SIZE))
        if frame_idx < X.shape[0]:
            if label in label_map:
                Y[frame_idx, label_map[label]] = 1
    
    # 5. Save as binary Numpy files
    base_name = filename.replace(".xml", "")
    np.save(f"{SAVE_DIR}/{base_name}_X.npy", X)
    np.save(f"{SAVE_DIR}/{base_name}_Y.npy", Y)

print("Success! All mixes processed and saved.")


SystemExit: Stopping cell

C:\Users\h4rsh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
